In [ ]:
import sys
sys.path.insert(0, '..')



# 02 — CF Families: Empirical Evidence for Non-Analytic Models

This notebook evaluates whether non-analytic characteristic-function (CF) families
fit financial returns better than the Gaussian baseline. The workflow mirrors the
paper: empirical CF visualization, AIC comparison, formal ECF-based goodness-of-fit
testing, rolling fit diagnostics, and tail behavior comparison.


## 1. Setup and data


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from cfad.simulate import gaussian_returns, nig_returns, levy_stable_returns
except Exception:
    from scipy.stats import levy_stable, norminvgauss

    def gaussian_returns(n: int, mu: float = 0.0, sigma: float = 0.01, seed: int | None = None):
        rng = np.random.default_rng(seed)
        return rng.normal(mu, sigma, int(n))

    def nig_returns(
        n: int,
        alpha: float = 10.0,
        beta: float = -0.5,
        delta: float = 0.12,
        mu: float = 0.0002,
        seed: int | None = None,
    ):
        return norminvgauss.rvs(
a=alpha,
b=beta,
loc=mu,
scale=delta,
size=int(n),
random_state=seed,
        )

    def levy_stable_returns(
        n: int,
        alpha: float = 1.7,
        beta: float = 0.0,
        c: float = 0.01,
        mu: float = 0.0,
        seed: int | None = None,
    ):
        return levy_stable.rvs(
alpha=alpha,
beta=beta,
loc=mu,
scale=c,
size=int(n),
random_state=seed,
        )

from cfad.models import GaussianCF, NIGCF, CGMYCF, LevyStableCF
from cfad.gof import aic_table, cf_distance, epps_pulley_test, rolling_gof
from cfad.empirical_cf import ecf_at
from cfad.utils import load_spy_sample


def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.22, linestyle=":", linewidth=0.7)
    ax.tick_params(labelsize=9)
    ax.xaxis.label.set_size(10)
    ax.yaxis.label.set_size(10)
    ax.title.set_size(10)


fig_dir = Path("../paper/figures")
fig_dir.mkdir(parents=True, exist_ok=True)

try:
    returns_series = load_spy_sample(start="2018-01-01", end="2023-01-01")
    returns_series = returns_series.dropna()
    dataset_name = "SPY log-returns"
except Exception as exc:
    warnings.warn(
        f"SPY download/cache unavailable ({exc}). Falling back to simulated NIG returns."
    )
    sim_vals = nig_returns(
        800,
        alpha=10.0,
        beta=-0.5,
        delta=0.12,
        mu=0.0002,
        seed=42,
    )
    sim_dates = pd.bdate_range("2019-01-01", periods=sim_vals.size)
    returns_series = pd.Series(sim_vals, index=sim_dates, name="log_return")
    dataset_name = "Simulated NIG fallback"

returns_series = returns_series.astype(float)
returns = returns_series.to_numpy(dtype=np.float64)
dates = pd.DatetimeIndex(returns_series.index)

print(f"Dataset: {dataset_name}")
print(f"n={len(returns)} | start={dates[0].date()} | end={dates[-1].date()}")
print(f"mean={returns.mean():.6f} | std={returns.std(ddof=1):.6f}")




## 2. Empirical CF plot

We compare the empirical CF with Gaussian and NIG fits over a shared frequency grid.
If NIG tracks both real and imaginary components more closely, that is direct evidence
that the return-generating law may carry non-analytic structure beyond the Gaussian null.


In [ ]:

xi = np.linspace(-12, 12, 400)
phi_hat = ecf_at(returns, xi)

gaussian_model = GaussianCF().fit(returns)
nig_model = NIGCF().fit(returns)

phi_gauss = gaussian_model.cf(xi)
phi_nig = nig_model.cf(xi)

branch_im = nig_model.alpha - abs(nig_model.beta)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)

axes[0].plot(xi, np.real(phi_hat), color="black", linewidth=1.6, label="Empirical CF")
axes[0].plot(
    xi,
    np.real(phi_gauss),
    color="tab:blue",
    linestyle="--",
    linewidth=1.4,
    label="Gaussian fit",
)
axes[0].plot(
    xi,
    np.real(phi_nig),
    color="tab:red",
    linewidth=1.4,
    label="NIG fit",
)
axes[0].set_title("Real part: Re[phi(xi)]")
axes[0].set_xlabel("xi")
axes[0].set_ylabel("Value")
style_ax(axes[0])
axes[0].legend(loc="upper right", fontsize=9, frameon=False)

axes[1].plot(xi, np.imag(phi_hat), color="black", linewidth=1.6, label="Empirical CF")
axes[1].plot(
    xi,
    np.imag(phi_gauss),
    color="tab:blue",
    linestyle="--",
    linewidth=1.4,
    label="Gaussian fit",
)
axes[1].plot(
    xi,
    np.imag(phi_nig),
    color="tab:red",
    linewidth=1.4,
    label="NIG fit",
)
axes[1].set_title("Imaginary part: Im[phi(xi)]")
axes[1].set_xlabel("xi")
axes[1].set_ylabel("Value")
style_ax(axes[1])
axes[1].legend(loc="upper right", fontsize=9, frameon=False)

axes[1].annotate(
    f"NIG branch point on imaginary axis\nxi = i({branch_im:.3f})",
    xy=(0.0, 0.0),
    xycoords="data",
    xytext=(0.03, 0.95),
    textcoords="axes fraction",
    fontsize=9,
    va="top",
    bbox=dict(facecolor="white", edgecolor="0.8", alpha=0.9),
    arrowprops=dict(arrowstyle="->", color="0.3", lw=1.0),
)

fig.suptitle("Empirical CF vs Gaussian and NIG fits", fontsize=11)
fig.tight_layout()
fig.savefig(fig_dir / "empirical_cf_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("Gaussian ECF-L2:", f"{cf_distance(returns, gaussian_model, xi_max=12, n_xi=400):.6f}")
print("NIG ECF-L2:", f"{cf_distance(returns, nig_model, xi_max=12, n_xi=400):.6f}")



## 3. AIC comparison table

A lower AIC indicates a better fit after penalizing model complexity.
If NIG (or other non-analytic families) wins, the data favors a structure that cannot be
represented by an entire Gaussian characteristic function.


In [ ]:
aic_df = aic_table(returns)

def highlight_winner(row):
    if bool(row["winner"]):
        return ["font-weight: bold; background-color: #f6f6d5"] * len(row)
    return ["" for _ in row]

display(aic_df.style.apply(highlight_winner, axis=1).format({"aic": "{:.4f}", "ecf_l2": "{:.6f}"}))

winner_name = aic_df.loc[aic_df["winner"], "model"].iloc[0]
print(f"Winner model: {winner_name}")



## 4. Goodness-of-fit tests

Epps-Pulley tests whether a model-implied CF can plausibly generate the observed data.
Rejecting Gaussian is not surprising for heavy-tailed market returns, but it is
structurally important because it indicates the entire-CF null is inadequate.


In [ ]:
gof_results = {}
for model_name, model_obj in [
    ("Gaussian", GaussianCF().fit(returns)),
    ("NIG", NIGCF().fit(returns)),
]:
    res = epps_pulley_test(returns, model_obj, B=499)
    gof_results[model_name] = res
    print(
        f"{model_name:8s} | statistic={res['statistic']:.4f} | "
        f"p-value={res['pvalue']:.4f} | reject_5pct={res['reject_5pct']}"
    )



## 5. CF distance across rolling windows


In [ ]:
window = 120
step = 5
rolling_l2 = rolling_gof(
    returns,
    GaussianCF,
    window=window,
    step=step,
    xi_max=8.0,
    n_xi=64,
)

mu_l2 = float(np.mean(rolling_l2))
sd_l2 = float(np.std(rolling_l2, ddof=1))
alert_thr = mu_l2 + 2.0 * sd_l2
is_high = rolling_l2 > alert_thr

end_idx = np.arange(window - 1, window - 1 + len(rolling_l2) * step, step)
end_idx = np.clip(end_idx, 0, len(returns) - 1)

x_l2 = dates[end_idx]
x_ret = dates

fig, ax1 = plt.subplots(figsize=(12, 4.8))
ax2 = ax1.twinx()

ax1.plot(x_l2, rolling_l2, color="tab:red", linewidth=1.4, label="Rolling Gaussian CF L2")
ax1.axhline(alert_thr, color="0.3", linestyle="--", linewidth=1.0, label="mean + 2σ")
ax1.scatter(
    x_l2[is_high],
    rolling_l2[is_high],
    color="red",
    s=20,
    zorder=4,
    label="Exceedance",
)
ax1.set_ylabel("Rolling CF distance", color="tab:red")
ax1.tick_params(axis="y", labelcolor="tab:red")
ax1.set_xlabel("Date")
style_ax(ax1)

ax2.plot(x_ret, returns, color="tab:blue", alpha=0.35, linewidth=1.0, label="Returns")
ax2.set_ylabel("Returns", color="tab:blue")
ax2.tick_params(axis="y", labelcolor="tab:blue")
ax2.spines["top"].set_visible(False)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=9, frameon=False)

ax1.set_title("Rolling Gaussian goodness-of-fit distance")
fig.tight_layout()
fig.savefig(fig_dir / "rolling_gof.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Rolling exceedances: {int(np.sum(is_high))} / {len(rolling_l2)} windows")



## 6. Tail comparison

On a log-log survival plot, slower tail decay indicates higher mass in extremes.
This tail behavior is consistent with non-analytic CF families, which encode jump-like
or heavy-tail structure that the Gaussian cannot represent.


In [ ]:
from scipy.stats import norminvgauss


def survival_abs(x: np.ndarray):
    x_abs = np.sort(np.abs(np.asarray(x, dtype=np.float64)))
    n = x_abs.size
    surv = 1.0 - (np.arange(1, n + 1) / n)
    surv = np.clip(surv, 1.0 / n, 1.0)
    return x_abs, surv


rng = np.random.default_rng(123)
gaussian_sim = rng.normal(gaussian_model.mu, gaussian_model.sigma, size=returns.size)
nig_sim = norminvgauss.rvs(
    a=nig_model.alpha,
    b=nig_model.beta,
    loc=nig_model.mu,
    scale=nig_model.delta,
    size=returns.size,
    random_state=456,
)

x_emp, y_emp = survival_abs(returns)
x_g, y_g = survival_abs(gaussian_sim)
x_n, y_n = survival_abs(nig_sim)

fig, ax = plt.subplots(figsize=(7.2, 5.0))
ax.loglog(x_emp, y_emp, color="black", linewidth=1.6, label="SPY / data")
ax.loglog(x_g, y_g, color="tab:blue", linestyle="--", linewidth=1.4, label="Gaussian sample")
ax.loglog(x_n, y_n, color="tab:red", linewidth=1.4, label="NIG sample")
ax.set_xlabel("|return|")
ax.set_ylabel("Survival: 1 - CDF")
ax.set_title("Tail comparison on log-log scale")
style_ax(ax)
ax.legend(loc="upper right", fontsize=9, frameon=False)
fig.tight_layout()
fig.savefig(fig_dir / "tail_comparison.png", dpi=150, bbox_inches="tight")
plt.show()



## 7. Save all figures


In [ ]:
required = [
    "empirical_cf_comparison.png",
    "rolling_gof.png",
    "tail_comparison.png",
]

existing = sorted([p.name for p in fig_dir.glob("*.png")])
print("Figure directory:", fig_dir.resolve())
print("Available PNG figures:")
for name in existing:
    print(" -", name)

missing = [name for name in required if name not in existing]
if missing:
    raise FileNotFoundError(f"Missing required figures: {missing}")
print("All required figures are present.")

